In [1]:
import pandas as pd
import pyomo.environ as pyo
from food_security import optimizer

from pathlib import Path

In [2]:
src_dir = Path('~').expanduser() / 'Stichting Deltares/International Delta Toolset - Documents (1)/Food security'

In [3]:
input_file = src_dir / 'input_all.xlsx'

crop_df = pd.read_excel(input_file, sheet_name='crop_id')
production_df = pd.read_excel(input_file, sheet_name='production_d')
production_df = production_df[production_df['year'] == 2020]
price_df = pd.read_excel(input_file, sheet_name='market_price')
price_df = price_df.fillna(180)
theta_df = pd.read_excel(input_file, sheet_name='plant_based_diet_goal')
energy_df = pd.read_excel(input_file, sheet_name='calories')
income_df = pd.read_excel(input_file, sheet_name='income')
price_goal_df = pd.read_excel(input_file, sheet_name='price_goal')

import_df = pd.read_excel(input_file, sheet_name='import')
export_df = pd.read_excel(input_file, sheet_name='export')

population = 106540000

In [4]:
print (import_df)

                                        crop_name_fao      import  2020  2030  \
0                                  Alfalfa for forage        0.00   NaN   NaN   
1   Animal products nec (Other animal products lik...        0.00   NaN   NaN   
2                                              Barley        0.03   NaN   NaN   
3                                          Beans, dry     1165.95   NaN   NaN   
4              Bovine cattle, sheep and goats, horses        0.00   NaN   NaN   
5                                            Cabbages        0.00   NaN   NaN   
6                                          Chick-peas        0.00   NaN   NaN   
7                                      Cotton linters        0.00   NaN   NaN   
8                              Cucumbers and gherkins        0.00   NaN   NaN   
9                              Eggplants (aubergines)        0.00   NaN   NaN   
10                                            Fishing        0.00   NaN   NaN   
11                          

In [5]:
trade_df = import_df[["crop_name_fao", "import"]].merge(
    export_df[["crop_name_fao", "export"]],
    on="crop_name_fao",
    how="outer",
)

In [6]:
income_df = income_df.merge(price_goal_df, on='income_quintiles')
income_df['p_spent_food_max'] = income_df['p_spent_food_max'] / 100

price_df = price_df.groupby(
    ["crop_name_fao"],
    as_index=False
)["market_price"].mean()

production_df = production_df.groupby(
    ["crop_name_fao"],
    as_index=False
)["yield"].sum()

trade_df = trade_df[trade_df['crop_name_fao'].isin(production_df['crop_name_fao'])]

energy_df = energy_df.groupby(
    ["crop_name_fao"],
    as_index=False
)["calories_kcal_kg"].mean()

input_df = (
    production_df
    .merge(price_df, on="crop_name_fao", how="left")
    .merge(trade_df, on="crop_name_fao", how="left")
    .merge(energy_df, on="crop_name_fao", how="left")
)

# Convert trade from tonnes to kg
input_df["import"] = input_df["import"] * 1000
input_df["export"] = input_df["export"] * 1000

input_df["quantity"] = input_df["yield"] + input_df["import"] - input_df["export"]

class_map = (
    crop_df
    .drop_duplicates(subset=["crop_name_fao"])
    .set_index("crop_name_fao")["class"]
)

input_df["class"] = input_df["crop_name_fao"].map(class_map)
input_df = input_df[input_df['class'] != '-']
input_df["quantity"] = input_df['quantity'].clip(lower=0, upper=None)
# Convert to daily availability
input_df["quantity"] = input_df["quantity"] / 365.25
input_df["market_price"] = input_df["market_price"] / 1000
input_df["usd_per_kg"] = input_df["market_price"]

In [7]:
FSOptimizer = optimizer.FSOptimizer(input_df=input_df, income_df=income_df, population=population, MDER=1806.61, lambda_g={
            g: 1.0 for g in ["carb", "protein", "fat", "fruitvege"]
        })
results = FSOptimizer.solve()

In [8]:
results[0]

,5,4,3,2,1
Barley,1.896976e+05,1.789416e+05,9.293991e+04,2.988145e+04,1.457131e+04
"Beans, dry",0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
Cabbages,7.435210e+06,4.801996e-01,1.743651e-02,5.008327e-03,2.503157e-03
Chick-peas,1.462113e+04,8.266226e-03,2.795111e-03,1.155864e-03,5.778313e-04
Cucumbers and gherkins,3.904230e+05,7.084795e-01,2.215429e-02,5.653448e-03,2.825462e-03
Eggplants (aubergines),1.259554e+07,4.337219e-01,1.599899e-02,4.771135e-03,2.384649e-03
"Groundnuts, Shelled",2.885960e+05,5.123690e-03,1.744105e-03,7.212813e-04,3.605779e-04
Lemons and limes,2.406574e+07,6.124042e-01,1.988758e-02,5.373243e-03,2.685475e-03
Linseed,9.597946e+04,5.916686e-03,2.007893e-03,8.304243e-04,4.151399e-04
Maize (incl. 0067 and 0068),2.192319e+05,7.164890e+06,3.868494e+06,7.843503e+05,3.735179e+05


In [9]:
results[1]

,5,4,3,2,1
Barley,0.008903,0.008398,0.004362,0.001402,0.000684
"Beans, dry",0.000000,0.000000,0.000000,0.000000,0.000000
Cabbages,0.348940,0.000000,0.000000,0.000000,0.000000
Chick-peas,0.000686,0.000000,0.000000,0.000000,0.000000
Cucumbers and gherkins,0.018323,0.000000,0.000000,0.000000,0.000000
Eggplants (aubergines),0.591118,0.000000,0.000000,0.000000,0.000000
"Groundnuts, Shelled",0.013544,0.000000,0.000000,0.000000,0.000000
Lemons and limes,1.129423,0.000000,0.000000,0.000000,0.000000
Linseed,0.004504,0.000000,0.000000,0.000000,0.000000
Maize (incl. 0067 and 0068),0.010289,0.336254,0.181551,0.036810,0.017529


In [10]:
results[2]

,5,4,3,2,1
objective function,0.372729,1.840000,1.840000,1.840000,1.840000
spending_pp,1.580570,0.282951,0.237189,0.226925,0.227783
percentage_spending,0.131714,0.031439,0.029649,0.037821,0.056946
consumed_energy_pp,1806.610018,1813.391279,1807.284293,1813.554997,1819.367521
carb_fraction,0.613131,1.000000,1.000000,1.000000,1.000000
fat_fraction,0.071085,0.000000,0.000000,0.000000,0.000000
fruitvege_fraction,0.306467,0.000000,0.000000,0.000000,0.000000
protein_fraction,0.009318,0.000000,0.000000,0.000000,0.000000
carb_kg,0.574074,0.586223,0.659500,0.723704,0.736661
fat_kg,0.024016,0.000000,0.000000,0.000000,0.000000
